In [3]:
import polars as pl
from src.storage.ch_client import ch_manager
import os

In [4]:
ch_client = ch_manager.market_db

sql = """
    SELECT 
        t.timestamp,
        t.symbol,
        t.price as trade_price,
        t.amount,
        (ob.bid_prices[1] * ob.ask_volumes[1] + ob.ask_prices[1] * ob.bid_volumes[1]) / nullIf(ob.bid_volumes[1] + ob.ask_volumes[1],0) as prev_micro_price,
        (ob.bid_volumes[1] - ob.ask_volumes[1]) / nullIf(ob.bid_volumes[1] + ob.ask_volumes[1],0) as imbalance,
        ob.bid_prices[1] as best_bid,
        ob.ask_prices[1] as best_ask,
        t.timestamp - ob.timestamp AS data_lag_ms
    FROM (
        SELECT * FROM market_data.trades 
        WHERE exchange_id = 'binance' AND symbol = 'BTC/USDT' 
        AND toDate(fromUnixTimestamp64Milli(timestamp)) = '2026-03-21'
    ) AS t
    ASOF LEFT JOIN (
        SELECT * FROM market_data.orderbook 
        WHERE exchange_id = 'binance' AND symbol = 'BTC/USDT' 
        AND toDate(fromUnixTimestamp64Milli(timestamp)) = '2026-03-21'
    ) AS ob
        ON t.exchange_id=ob.exchange_id
        AND t.symbol=ob.symbol
        AND t.timestamp >= ob.timestamp
"""

arrow_table = ch_client.query_arrow(sql)

df = pl.from_arrow(arrow_table)

del arrow_table

clear_df = df.filter(pl.col('data_lag_ms').abs() <= 100)

impact_df = clear_df.with_columns([
    ((pl.col('best_ask') - pl.col('trade_price')) / pl.col('best_ask') * 1000).alias('slippage_bps_buy'),
    ((pl.col('trade_price') - pl.col('best_bid')) / pl.col('best_bid') * 1000).alias('slippage_bps_sell'),
    (pl.col('trade_price') * pl.col('amount')).alias('notional_usdt')
])

size_buckets = impact_df.with_columns([
    pl.when(pl.col('notional_usdt') < 1000).then(pl.lit('Retail (<1k)')).when(pl.col('notional_usdt') < 10000).then(pl.lit('Pro (1k-10k)')).otherwise(pl.lit('Whale (>10k)')).alias('trader_category')
]).group_by('trader_category').agg([
    pl.col('slippage_bps_buy').mean().alias('avg_impact_cost_buy'),
    pl.col('slippage_bps_sell').mean().alias('avg_impact_cost_sell'),
    pl.len().alias('trade_count')
])

print(size_buckets)


shape: (3, 4)
┌─────────────────┬─────────────────────┬──────────────────────┬─────────────┐
│ trader_category ┆ avg_impact_cost_buy ┆ avg_impact_cost_sell ┆ trade_count │
│ ---             ┆ ---                 ┆ ---                  ┆ ---         │
│ str             ┆ f64                 ┆ f64                  ┆ u32         │
╞═════════════════╪═════════════════════╪══════════════════════╪═════════════╡
│ Whale (>10k)    ┆ -0.014163           ┆ 0.016004             ┆ 776         │
│ Pro (1k-10k)    ┆ -0.019252           ┆ 0.021855             ┆ 2830        │
│ Retail (<1k)    ┆ -0.009811           ┆ 0.010922             ┆ 68774       │
└─────────────────┴─────────────────────┴──────────────────────┴─────────────┘


In [ ]:
ob_sql = """
    SELECT
        timestamp,
        symbol,
        bid_prices,
        bid_volumes,
        ask_prices,
        ask_volumes,
        (bid_prices[1] + ask_prices[1]) / 2 as mid_price
    FROM market_data.orderbook
    WHERE exchange_id='binance' 
    AND symbol='BTC/USDT'
    AND toDate(fromUnixTimestamp64Milli(timestamp)) = '2026-03-22'
    ORDER BY timestamp ASC
    LIMIT 100000
"""

ob_arrow = ch_client.query_arrow(ob_sql)
ob_df = pl.from_arrow(ob_arrow)